In [ ]:
import httpx
import os
from bs4 import BeautifulSoup
import asyncio
import random
from user_agents import get_random_user_agent, get_random_header
from urllib.parse import urlparse

In [ ]:
BASE_URL = 'https://medlineplus.gov/'
OUTPUT_FOLDER = 'corpus/medline/'

In [ ]:
async def get_html(url):
    async with httpx.AsyncClient() as client:
        headers = get_random_header()
        try:
            await asyncio.sleep(random.random() + 1) 
            response = await client.get(url, follow_redirects=True, headers = headers, timeout=60.0)
            if response.status_code == 200:
                return response.text
            else:
                return response
        except Exception as e:
            print(repr(e), url)
            return None
    
async def writeMedData(med_url, filename):
    med_page = await get_html(med_url)
    if med_page is not None:
        med_soup = BeautifulSoup(med_page, 'html.parser')
        med_title = med_soup.find("h1", attrs={"itemprop": "name"})
        med_sum = med_soup.find("div", id="topic-summary")
        f = open(OUTPUT_FOLDER + filename, "w")
        if med_title is not None:
            med_txt = med_title.getText()
            lines = [line.strip() for line in med_txt.split('\n') if line.strip()]
            for l in lines:
                f.write(l + "\n")
            
        if med_sum is not None:
            med_txt = med_sum.getText()
            lines = [line.strip() for line in med_txt.split('\n') if line.strip()]
            for l in lines:
                f.write(l + "\n")
        f.close()

In [ ]:
with open("medline_list.html") as tf:
    n = 0
    med_page = tf.read()
    med_soup = BeautifulSoup(med_page, 'html.parser')
    li_list = med_soup.findAll("li", class_="item")
    
    for li in li_list:
        an = li.findAll("a")
        for a in an:
            cod = f'MDL{n:05}'
            fname_es = cod + '_es.txt'
            fname_en = cod + '_en.txt'

            url_es = BASE_URL + 'spanish' + urlparse(a["href"]).path
            url_en = a["href"]

            await writeMedData(url_es, fname_es)
            print(fname_es, url_es)
            await writeMedData(url_en, fname_en)
            print(fname_en, url_en)
            n += 1
        
    print("FIN")
        

In [ ]:
import os

PATH = "corpus/medline/"

n = 0
for cont in range(2488):
    code = f'MDL{cont:05}'
    if not os.path.exists(PATH + code + '_es.txt'):
        print(code, 'es')
        n += 1
    if not os.path.exists(PATH + code + '_en.txt'):
        print(code, 'en')
        n += 1
print("Total:", n)